# K4P: profile the training step (diagnostic)

Generated by `scripts/make_notebooks.py`; see `docs/kaggle_runbook.md`. Run it headless:
**Save Version -> Save & Run All (Commit)**. No cell needs editing.

- **Accelerator:** **GPU T4 x2**. **Internet:** ON.
- **Attach (Add Input):** the K2 output for SIZE
- **Expected time:** ~10-15 min (two configs, ~450 steps each, cold start included)
- **Output (`/kaggle/working/out`):** `profile/<name>/profile.json` and `summary.txt` per config, marker `_HOOKREC_PROFILE`

Runs `scripts/profile_train.py` on each CONFIG: per-phase step times (data wait, H2D, encode, negatives + item_repr, loss, backward, optimizer), the unsynchronised steps/s, a torch.profiler top-25, one timed periodic eval and a DataLoader-only sweep (num_workers 0/2/4 x pin_memory). Nothing is trained to keep.

GIT_REF defaults to the release tag the K4 runs use, so the profiled step is the step those runs train.

In [ ]:
# Parameters: the defaults work unedited (attach inputs, then Save Version ->
# Save & Run All). Generated by scripts/make_notebooks.py.
REPO_URL = "https://github.com/Arubad/Reccomendation_system.git"  # the HookSeq repo (public: no token needed; GITHUB_TOKEN only for a private fork)
GIT_REF = "v0.4"  # branch or tag to run (pinned to the release tag of the reported runs)
SIZE = "50m"  # Yambda size: '50m' or '500m'
CONFIGS = ["configs/sasrec.yaml", "configs/hookseq.yaml"]  # profiled in order
STEPS = 300  # phased (synchronised) steps per config
WARMUP = 20  # timed warmup steps (cold start is reported from these)
THROUGHPUT_STEPS = 100  # unsynchronised steps (the real steps/s)
PROFILE_STEPS = 20  # torch.profiler steps
LOADER_WORKERS = "0,2,4"  # num_workers values of the DataLoader-only sweep
OVERRIDES = []  # applied to every config, e.g. ["loss.chunk_rows=4096"]
REQUIRE_GPU = True  # fail fast if the GPU accelerator is not on

# fixed Kaggle paths
INPUT_ROOT = "/kaggle/input"
WORK = "/kaggle/working"
OUT = "/kaggle/working/out"
REPO = "/kaggle/working/repo"

import os
import time

# train.max_hours is measured from here (the session start), not from the script start
os.environ.setdefault("HOOKREC_SESSION_START", str(int(time.time())))
print("HOOKREC_SESSION_START =", os.environ["HOOKREC_SESSION_START"])

In [ ]:
# Clone the repo and install it. The image's torch is used (no torch install).
import importlib
import os
import shutil
import subprocess
import sys


import base64
import re
import threading
from importlib import metadata
from urllib.parse import urlsplit


def _stream(cmd, timeout):
    """Run cmd, streaming its output; raise on a non-zero exit (a `!cmd` magic would not)."""
    p = subprocess.Popen(
        cmd, stdin=subprocess.DEVNULL, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
        text=True, bufsize=1,
    )
    expired = []
    timer = threading.Timer(timeout, lambda: (expired.append(1), p.kill()))
    timer.start()
    try:
        for line in p.stdout:
            print(line, end="", flush=True)
    finally:
        timer.cancel()
        p.stdout.close()
    code = p.wait()
    if expired:
        raise RuntimeError(f"{' '.join(cmd[:5])} ... timed out after {timeout} s (network?)")
    if code != 0:
        raise RuntimeError(f"{' '.join(cmd[:5])} ... exited {code}")


if os.path.exists(REPO):
    shutil.rmtree(REPO)
token = basic = u = ""
try:  # optional, only for a private fork: Add-ons -> Secrets -> GITHUB_TOKEN (attached)
    from kaggle_secrets import UserSecretsClient

    token = UserSecretsClient().get_secret("GITHUB_TOKEN") or ""
except Exception:
    token = ""
# never prompt (a headless run would hang); the token travels only in this process's env
# as an HTTP header for REPO_URL's host: never in argv, the URL, .git/config or .git/logs
git_env = {**os.environ, "GIT_TERMINAL_PROMPT": "0", "GCM_INTERACTIVE": "never"}
if token:
    u = urlsplit(REPO_URL)
    basic = base64.b64encode(f"x-access-token:{token}".encode()).decode()
    git_env.update(
        GIT_CONFIG_COUNT="1",
        GIT_CONFIG_KEY_0=f"http.{u.scheme}://{u.netloc}/.extraheader",
        GIT_CONFIG_VALUE_0=f"Authorization: Basic {basic}",
    )
try:
    r = subprocess.run(
        ["git", "clone", "--depth", "1", "-b", GIT_REF, REPO_URL, REPO],
        env=git_env, stdin=subprocess.DEVNULL, capture_output=True, text=True, timeout=600,
    )
except subprocess.TimeoutExpired:
    raise RuntimeError(f"git clone {REPO_URL} -b {GIT_REF} timed out after 600 s") from None
if r.returncode != 0:
    log = r.stdout + r.stderr
    if token:
        log = log.replace(token, "***").replace(basic, "***")
    raise RuntimeError(
        f"git clone {REPO_URL} -b {GIT_REF} failed (exit {r.returncode}):\n{log}\n"
        "Check REPO_URL / GIT_REF and that Internet is ON; a private fork needs the "
        "GITHUB_TOKEN secret attached to this notebook."
    )
del token, git_env, r, basic, u
_stream([sys.executable, "-m", "pip", "install", "-q", "-e", REPO], timeout=1800)
sys.path.insert(0, os.path.join(REPO, "src"))
importlib.invalidate_caches()
# the installed (not an already-imported, possibly stale) polars must meet pyproject's floor
pl_version = metadata.version("polars")
if tuple(int(x) for x in re.findall(r"\d+", pl_version)[:2]) < (2, 1):
    raise RuntimeError(
        f"polars {pl_version} is installed but hookrec needs polars>=2.1 (pl.Float16, "
        "streaming collect); pip did not upgrade it. Check the pip log above."
    )
SHA = subprocess.run(
    ["git", "-C", REPO, "rev-parse", "HEAD"], capture_output=True, text=True, check=True
).stdout.strip()
print(f"hookrec {REPO_URL} {GIT_REF} @ {SHA} (polars {pl_version})")

In [ ]:
# Environment + input discovery (inputs are found by marker files; no slugs to edit).
import os
import sys
from pathlib import Path

from hookrec.utils.kaggle import (
    check_disk, env_report, find_inputs, find_one, find_processed, has_run, run_logged,
)

env_report(WORK)

import torch

if REQUIRE_GPU and not torch.cuda.is_available():
    raise RuntimeError(
        "no GPU: set Accelerator = GPU T4 x2 in the notebook sidebar (Settings), then Save "
        "Version again. (REQUIRE_GPU = False profiles on CPU, which says little about the GPU.)"
    )
if torch.cuda.is_available():
    cap = torch.cuda.get_device_capability()
    print("torch", torch.__version__, "cuda", torch.version.cuda, "device sm_%d%d" % cap,
          "arch list", torch.cuda.get_arch_list())
PROCESSED = find_processed(SIZE, INPUT_ROOT)
print("processed:", PROCESSED)

In [ ]:
import sys

PROFILES = Path(OUT, "profile")
for config in CONFIGS:  # one fresh process per config, as K4 trains them
    name = Path(config).stem
    cmd = [
        sys.executable, f"{REPO}/scripts/profile_train.py", "--processed", PROCESSED,
        "--config", os.path.join(REPO, config), "--steps", STEPS, "--warmup", WARMUP,
        "--throughput-steps", THROUGHPUT_STEPS, "--profile-steps", PROFILE_STEPS,
        "--loader-workers", LOADER_WORKERS, "--out", PROFILES / name,
    ]
    for o in OVERRIDES:
        cmd += ["--override", o]
    run_logged(cmd, cwd=REPO)
(PROFILES / "_HOOKREC_PROFILE").write_text("")

In [ ]:
check_disk(WORK)
for config in CONFIGS:
    name = Path(config).stem
    print("=" * 30, name, "=" * 30)
    print((PROFILES / name / "summary.txt").read_text())